# Test Silver — INE
Verifica las 5 tablas Silver de INE sin escanear datos completos.
Usa `limit(10)` — no ejecuta `count()` ni agregaciones para no gastar créditos.

In [ ]:
SILVER_SCHEMA = "stage_silver_ss2"

_EDAD_TABLES = [
    "ine_defunciones_sexo_edad",
    "ine_defunciones_neonatales",
    "ine_defunciones_postneonatales",
]

_GEO_TABLES = [
    "ine_defunciones_depto_residencia",
    "ine_defunciones_causas_externas",
]

---
## Tablas INE — Edad

### ine_defunciones_sexo_edad

In [ ]:
df = spark.table(f"{SILVER_SCHEMA}.ine_defunciones_sexo_edad")
df.printSchema()

In [ ]:
display(df.limit(10))

### ine_defunciones_neonatales

In [ ]:
df = spark.table(f"{SILVER_SCHEMA}.ine_defunciones_neonatales")
df.printSchema()

In [ ]:
display(df.limit(10))

### ine_defunciones_postneonatales

In [ ]:
df = spark.table(f"{SILVER_SCHEMA}.ine_defunciones_postneonatales")
df.printSchema()

In [ ]:
display(df.limit(10))

---
## Tablas INE — Geografía

### ine_defunciones_depto_residencia

In [ ]:
df = spark.table(f"{SILVER_SCHEMA}.ine_defunciones_depto_residencia")
df.printSchema()

In [ ]:
display(df.limit(10))

### ine_defunciones_causas_externas

In [ ]:
df = spark.table(f"{SILVER_SCHEMA}.ine_defunciones_causas_externas")
df.printSchema()

In [ ]:
display(df.limit(10))

---
## Chequeos rápidos de calidad (sin full scan)
Lee solo las primeras 1000 filas de cada tabla para verificar transformaciones clave.

In [ ]:
from pyspark.sql import functions as F

print("── Tablas INE Edad ──")
for t in _EDAD_TABLES:
    sample = spark.table(f"{SILVER_SCHEMA}.{t}").limit(1000)

    anio_nulls  = sample.filter(F.col("anio").isNull()).count()
    cie_nulls   = sample.filter(F.col("cie_10_norm").isNull()).count()
    cie_sucio   = sample.filter(F.col("cie_10_norm").rlike(r'[: ]')).count()

    print(f"\n  {t}")
    print(f"    anio nulls          : {anio_nulls}")
    print(f"    cie_10_norm nulls   : {cie_nulls}")
    print(f"    cie_10_norm sucio   : {cie_sucio}  (debe ser 0)")

print("\n── Tablas INE Geografía ──")
for t in _GEO_TABLES:
    sample = spark.table(f"{SILVER_SCHEMA}.{t}").limit(1000)

    anio_nulls   = sample.filter(F.col("anio").isNull()).count()
    cie_nulls    = sample.filter(F.col("cie_10_norm").isNull()).count()
    cie_sucio    = sample.filter(F.col("cie_10_norm").rlike(r'[: ]')).count()
    depto_nulls  = sample.filter(F.col("departamento_oficial").isNull()).count()
    depto_cols   = [c for c in spark.table(f"{SILVER_SCHEMA}.{t}").columns
                    if "departamento_de" in c]
    depto_col    = depto_cols[0] if depto_cols else None
    subtotal_leak = (
        sample.filter(F.upper(F.trim(F.col(depto_col))) == "TODOS LOS DEPARTAMENTOS").count()
        if depto_col else "N/A"
    )

    print(f"\n  {t}")
    print(f"    anio nulls          : {anio_nulls}")
    print(f"    cie_10_norm nulls   : {cie_nulls}")
    print(f"    cie_10_norm sucio   : {cie_sucio}  (debe ser 0)")
    print(f"    depto sin match     : {depto_nulls}")
    print(f"    subtotales filtrados: {subtotal_leak}  (debe ser 0)")

# causas_externas tiene causa_de_muerte derivada del campo combinado Bronze
# Verificar que la separación fue correcta: sin paréntesis, sin nulls
print("\n── causas_externas: separación causa_de_muerte / cie_10_norm ──")
_ce = spark.table(f"{SILVER_SCHEMA}.ine_defunciones_causas_externas").limit(1000)
causa_nulls     = _ce.filter(F.col("causa_de_muerte").isNull()).count()
causa_con_paren = _ce.filter(F.col("causa_de_muerte").rlike(r'\(')).count()
cie_con_texto   = _ce.filter(F.col("cie_10_norm").rlike(r'[a-záéíóúñ ]')).count()
print(f"  causa_de_muerte nulls       : {causa_nulls}  (debe ser 0)")
print(f"  causa_de_muerte con '('     : {causa_con_paren}  (debe ser 0 — paréntesis ya extraídos)")
print(f"  cie_10_norm con texto/spaces: {cie_con_texto}  (debe ser 0 — solo rangos como V01-V99)")
campo_null = _ce.filter(F.col("causas_externas_codigo_cie_10").isNull()).count()
print(f"  causas_externas_codigo nulos: {campo_null}  (debe ser 0 — filtradas en Silver)")